# A FAIR²-Compliant Dataset of Global Library and Information Science Journals Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²-Compliant Dataset of Global Library and Information Science Journals](https://sen.science/doi/10.71728/senscience.xjp1-3zhc/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset is provided as a Croissant schema via the following URL:

`https://sen.science/doi/10.71728/senscience.xjp1-3zhc/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.xjp1-3zhc/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# .metadata is a croissant.Metadata object
meta = dataset.metadata

# Print dataset title and description
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets and their fields, referencing all by `@id`.

### List Record Sets
First, inspect the available record sets. Each record set is referenced by its unique `@id`.

In [ ]:
# List available record sets and their field IDs
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in this dataset. Please check the schema or contact the dataset curator.")
else:
    for rs in record_sets:
        print(f'RecordSet @id: {rs.id}')
        print(f'  Name: {getattr(rs, "name", "N/A")}')
        print(f'  Description: {getattr(rs, "description", "N/A")}')
        print("  Fields:")
        for field in rs.fields:
            print(f'    Field @id: {field.id}')
            print(f'      Name: {getattr(field, "name", "N/A")}')
            print(f'      DataType: {getattr(field, "data_type", "N/A")}')
            print(f'      Description: {getattr(field, "description", "N/A")}')
        print("---")

### Record Example
For each record set, print examples of records. Each record set and field is referenced by its `@id`.

*If there are multiple record sets, you can select one of interest by its `@id`.*

In [ ]:
# Print a few records for each record set
for rs in dataset.record_sets:
    print(f"RecordSet @id: {rs.id}")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs.id)):
            print(f"Record {i+1}: {rec}")
            if i>=2:
                break
    except Exception as e:
        print(f"Could not load records for RecordSet {rs.id}: {e}")
    print("---")

## 3. Data Extraction
Load records from specific record sets into Pandas DataFrames for analysis. All references use the `@id` of the record set and fields.

Below, we select all record sets available and load them into DataFrames. If the set is empty, return a notification.

In [ ]:
# Extract all data from each record set using their @id
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rsid in record_set_ids:
    try:
        recs = list(dataset.records(record_set=rsid))
        if recs:
            df = pd.DataFrame(recs)
            dataframes[rsid] = df
            print(f"Loaded RecordSet @id: {rsid} with shape {df.shape}")
            print("Columns:", df.columns.tolist())
        else:
            print(f"No records for RecordSet @id: {rsid}")
    except Exception as e:
        print(f"Failed to load RecordSet {rsid}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filter records based on specific criteria.
- Normalize numeric fields.
- Categorize or group data.

**All operations below reference the record set and fields by their `@id`.**
First, select a record set and numeric field by their `@id` from the loaded DataFrames above. Update the cell below with the actual IDs shown from the previous step.

In [ ]:
# Choose a record set and numeric field by their @id for EDA
# For demonstration, we'll select the first loaded record set and first numeric column
if dataframes:
    selected_rs_id = next(iter(dataframes))
    df = dataframes[selected_rs_id]
    numeric_fields = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_fields.append(col)
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use @id as column name
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in RecordSet @id: {selected_rs_id} where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Grouping by a categorical field
        # Find a non-numeric column
        group_field = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped normalized records by categorical field {group_field}:")
            print(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print(f"No numeric fields found in RecordSet @id: {selected_rs_id}.")
else:
    print("No dataframes loaded to perform EDA.")

## 5. Visualization
Visualize distributions or relationships in the dataset using the selected numeric field and categorical grouping.

If available, use the selected fields by their `@id` for all plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field in selected RecordSet
if dataframes:
    df = dataframes[selected_rs_id]
    if 'numeric_field_id' in locals():
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field_id} in RecordSet @id: {selected_rs_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
        if group_field:
            plt.figure(figsize=(10,6))
            sns.boxplot(x=group_field, y=numeric_field_id, data=df)
            plt.title(f"{numeric_field_id} by {group_field} in RecordSet @id: {selected_rs_id}")
            plt.xticks(rotation=45)
            plt.show()
    else:
        print("No numeric field identified for plotting.")
else:
    print("No dataframes loaded for visualization.")

## 6. Conclusion
This notebook guided you through loading and exploring a Croissant dataset using the `mlcroissant` library. We loaded metadata, listed record sets and fields by their `@id`, loaded all available records, performed basic exploratory data analysis and normalization, grouped using categorical fields, and visualized distributions. You can extend this notebook by adding more complex analyses, joining record sets, or leveraging the rich metadata.

**All processing referenced entities by their `@id`, ensuring reproducibility and clarity.**